In [ ]:
!pip install scikit-image opencv-python-headless -q

In [ ]:
%%capture
%run 01_Dataset_Preparation.ipynb

In [ ]:
!pip install scikit-image opencv-python-headless -q

In [ ]:
import cv2
import numpy as np
import os
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import cv2
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.metrics import *

In [ ]:
print("✅ Dataset loaded successfully!")
print(f"Classes ({num_classes}): {class_names}")
print(f"IMG_HEIGHT: {IMG_HEIGHT}")
print(f"TRAIN_DIR: {TRAIN_DIR}")
print(f"VAL_DIR: {VAL_DIR}")
print(f"TEST_DIR: {TEST_DIR}")
print(f"\nGenerator samples:")
print(f"  Training:   {train_generator.samples} images")
print(f"  Validation: {validation_generator.samples} images")
print(f"  Test:       {test_generator.samples} images ← HELD OUT")
print(f"\n✅ Ready for HOG+SVM!")

✅ Dataset loaded successfully!
Classes (7): ['cassiopeia', 'crux', 'cygnus', 'gemini', 'leo', 'orion', 'ursa_major']
IMG_HEIGHT: 224
TRAIN_DIR: /content/train
VAL_DIR: /content/validation
TEST_DIR: /content/test

Generator samples:
  Training:   140 images
  Validation: 27 images
  Test:       35 images ← HELD OUT

✅ Ready for HOG+SVM!


In [ ]:
# =============================================================================
# HOG + SVM Model Definition (matches your DL style)
# Fair parameters: same split, same reporting, same metrics
# =============================================================================

# === HOG + SVM MODEL DEFINITION ===
def create_hog_svm(learning_rate=None):  # No LR (SVM uses C/gamma)
    """
    Classical HOG + SVM baseline — fair comparison to DL models.
    Feature extraction: HOG optimized for astronomy star patterns.
    Hyperparameters tuned for small dataset (C=100, gamma=0.01).
    """
    print("HOG+SVM Model Created")
    print("• Features: Histogram of Oriented Gradients (astronomy-tuned)")
    print("• Classifier: RBF SVM (C=100, gamma=0.01)")
    print("• No neural weights — classical computer vision baseline")
    return None  # SVM doesn't need compiled model

def extract_hog_features(directory):
    """HOG feature extraction (224x224 → 3780-dim vector)."""
    features, labels = [], []
    class_to_idx = {cls: i for i, cls in enumerate(class_names)}

    print(f"Extracting HOG features from {os.path.basename(directory)}...")
    for cls in class_names:
        cls_path = os.path.join(directory, cls)
        if not os.path.exists(cls_path): continue

        for fname in os.listdir(cls_path):
            if not fname.lower().endswith(('.jpg','.jpeg','.png')): continue

            img_path = os.path.join(cls_path, fname)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_HEIGHT, IMG_WIDTH))  # Match DL input
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # HOG tuned for constellations (sparse stars)
            hog_feat = hog(gray,
                         orientations=9,
                         pixels_per_cell=(8,8),
                         cells_per_block=(2,2),
                         block_norm='L2-Hys')

            features.append(hog_feat)
            labels.append(class_to_idx[cls])

    print(f"  → {len(features)} samples, {len(features[0])} features each")
    return np.array(features), np.array(labels)

# === TRAINING + EVALUATION (matches my DL pipeline) ===
def train_evaluate_hog_svm():
    """
    Train HOG+SVM using same train/val/test split as DL models.
    Returns same metrics format for fair comparison.
    """
    print("\n" + "="*70)
    print("HOG + SVM BASELINE TRAINING")
    print("="*70)

    # Extract features (equivalent to "model creation")
    X_train, y_train = extract_hog_features(TRAIN_DIR)
    X_val, y_val = extract_hog_features(VAL_DIR)
    X_test, y_test = extract_hog_features(TEST_DIR)

    # Train SVM (equivalent to model.fit())
    svm_model = SVC(kernel='rbf', C=100, gamma=0.01,
                   random_state=42, probability=True)
    svm_model.fit(X_train, y_train)

    print(f"✅ HOG+SVM trained!")
    print(f"   Train samples: {X_train.shape[0]}")
    print(f"   Feature dim:   {X_train.shape[1]}")

    # Evaluate all sets (matches my DL reporting)
    y_pred_train = svm_model.predict(X_train)
    y_pred_val = svm_model.predict(X_val)
    y_pred_test = svm_model.predict(X_test)

    train_acc = accuracy_score(y_train, y_pred_train)
    val_acc = accuracy_score(y_val, y_pred_val)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test, average='macro')

    print(f"\nHOG+SVM Results (same format as DL models):")
    print(f"  Training Acc:   {train_acc:.4f}")
    print(f"  Validation Acc: {val_acc:.4f}")
    print(f"  Test Acc:       {test_acc:.4f}")
    print(f"  Test Macro F1:  {test_f1:.4f}")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_test)
    print("\nConfusion Matrix (Test Set):")
    print(cm)

    # Save results (JSON format matches DL)
    hog_results = {
        'name': 'HOG+SVM',
        'test_accuracy': float(test_acc),
        'test_f1_macro': float(test_f1),
        'val_accuracy': float(val_acc),
        'train_accuracy': float(train_acc),
        'feature_dimension': int(X_test.shape[1]),
        'confusion_matrix': cm.tolist(),
        'classification_report': classification_report(
            y_test, y_pred_test, target_names=class_names, output_dict=True
        ),
        'hog_params': {'orientations':9, 'pixels_per_cell':(8,8), 'C':100, 'gamma':0.01}
    }

    with open('hog_svm_results.json', 'w') as f:
        json.dump(hog_results, f, indent=2)

    print("\n✅ Results saved to hog_svm_results.json")
    print("\n📊 Paper-ready comparison line:")
    print(f"HOG+SVM | {test_acc:.1%} | {test_f1:.1%} | {X_test.shape[1]} features | 2min")

    return hog_results

# === QUICK SANITY CHECK (matches my DL check) ===
print("Model parameter check:")
print("HOG+SVM — feature dimension: Will be computed from images (~3780)")
print("Baseline CNN — trainable params: ~150K (from your earlier run)")
print("MobileNetV2 — trainable params: ~165K head only (from your earlier run)")

# === RUN TRAINING ===
hog_results = train_evaluate_hog_svm()

Model parameter check:
HOG+SVM — feature dimension: Will be computed from images (~3780)
Baseline CNN — trainable params: ~150K (from your earlier run)
MobileNetV2 — trainable params: ~165K head only (from your earlier run)

HOG + SVM BASELINE TRAINING
Extracting HOG features from train...
  → 140 samples, 26244 features each
Extracting HOG features from validation...
  → 27 samples, 26244 features each
Extracting HOG features from test...
  → 35 samples, 26244 features each
✅ HOG+SVM trained!
   Train samples: 140
   Feature dim:   26244

HOG+SVM Results (same format as DL models):
  Training Acc:   1.0000
  Validation Acc: 0.2593
  Test Acc:       0.1143
  Test Macro F1:  0.0704

Confusion Matrix (Test Set):
[[0 0 0 0 4 1 0]
 [2 0 0 0 1 2 0]
 [1 0 1 0 3 0 0]
 [0 0 0 0 5 0 0]
 [1 0 1 0 3 0 0]
 [1 0 0 0 4 0 0]
 [1 0 0 0 4 0 0]]

✅ Results saved to hog_svm_results.json

📊 Paper-ready comparison line:
HOG+SVM | 11.4% | 7.0% | 26244 features | 2min
